# Preparando o ambiente da Arena Analytics

## Demanda da aula 1

**Quem pediu:** o líder técnico do time de dados da Arena Analytics.

> "Decidimos usar Spark neste projeto. Hoje o histórico das Copas é pequeno, mas o projeto vai crescer: além do
> histórico, também vamos receber os eventos de cada partida, lance a lance, e uma Copa nova a cada quatro anos.
> Para começar, suba o ambiente e leia os dados do fornecedor."

**Contexto:** o fornecedor manda três arquivos CSV com o histórico das Copas. Um protótipo anterior quebrou quando
chegou um arquivo num formato diferente do esperado.

| Etapa | O que fazer | Notebook |
|---|---|---|
| Task 1 | Subir o ambiente do projeto, igual em qualquer máquina | `02_preparando_o_ambiente` |
| Conhecendo o Spark | O que é o Spark e por que ele aguenta o crescimento do projeto | `03_introducao_ao_spark` |
| Conhecendo o Spark | O primeiro cálculo e como o Spark executa por dentro | `04_como_o_spark_funciona` |
| Task 2 | Ler os três arquivos do fornecedor com um schema declarado no projeto | `05_lendo_os_primeiros_arquivos` |

**Pronto quando:** o ambiente sobe com um comando, e os três arquivos são lidos com um contrato (schema) que mora no
código do projeto.

## Task 1: subir o ambiente do projeto

### Container

Instalar o Spark à mão exige Java e Python nas versões certas, variáveis de ambiente e, no Windows, ajustes extras.
Um **container Docker** resolve isso: um ambiente isolado, com tudo instalado nas mesmas versões, igual em qualquer
sistema operacional.

- **Imagem:** a receita do ambiente (sistema, programas e versões).
- **Container:** a imagem rodando.

```
seu computador (Windows, Mac ou Linux)
  └── container do curso
        ├── Java        (o Spark roda na JVM)
        ├── Python      (onde você escreve o código)
        ├── Spark 4.2   (o motor de processamento)
        ├── Jupyter     (onde ficam os notebooks)
        └── /home/jovyan/work  =  a pasta do projeto no seu computador
```

A pasta do projeto é **compartilhada**: o que é salvo no container aparece no seu computador, e vice-versa.

### Os arquivos do projeto

| Arquivo ou pasta | Para que serve |
|---|---|
| `Dockerfile` | a receita da imagem: parte de uma imagem pronta com Jupyter e Spark, com a versão fixada, e acrescenta as bibliotecas do curso |
| `docker-compose.yml` | como subir o container: as portas (8888 para o Jupyter, 4040 a 4045 para a Spark UI), a pasta compartilhada e a senha do Jupyter |
| `README.md` | o passo a passo para subir o ambiente |
| `data/` | os dados que o fornecedor manda: o histórico das Copas, o feed de eventos e a Copa de 2026 |
| `src/wc_pipeline/` | o código do projeto. Começa só com os schemas e cresce conforme o projeto avança |
| `notebooks/` | os notebooks de cada aula |

### Subir, derrubar e subir de novo

No **terminal do seu computador**, dentro da pasta do projeto:

| Quando | Comando | O que faz |
|---|---|---|
| a primeira vez | `docker compose up --build` | monta a imagem a partir do `Dockerfile` e sobe o container |
| para parar | `Ctrl+C` e depois `docker compose down` | para e remove o container; a imagem e os arquivos continuam |
| das próximas vezes | `docker compose up` | sobe o container a partir da imagem que já existe |
| se o `Dockerfile` mudar | `docker compose up --build` | monta a imagem de novo |

Jupyter: http://localhost:8888/lab?token=estudo

### SparkSession

A **SparkSession** é a porta de entrada do Spark: é por ela que o código Python pede ao Spark para ler arquivos,
fazer contas e mostrar resultados. Criar a sessão liga o Spark para o notebook, e `spark.stop()` desliga. Por
convenção, ela fica na variável `spark`.

| Linha | O que faz |
|---|---|
| `.appName(...)` | dá um nome ao programa |
| `.master("local[*]")` | roda aqui no container (`local`), usando todos os núcleos do processador (`[*]`) |
| `.config(...)` | um ajuste; este define onde o Spark guarda as tabelas que ele gerencia |
| `.getOrCreate()` | cria a sessão, ou reaproveita a que já existe no notebook |

In [ ]:
# crie a SparkSession e imprima spark.version


### Spark, PySpark e Java

O Spark é escrito em **Scala**, que roda na **JVM** (a máquina virtual do Java). O **PySpark** é a porta de entrada
em Python:

```
seu código Python  ->  PySpark  ->  Spark (na JVM)  ->  resultado volta para o Python
```

O `!` no começo da célula executa o comando no terminal do container, e não no Python.

In [ ]:
# rode java -version no terminal do container com !


### Primeiro DataFrame

`spark.range(5)` cria uma tabela com uma coluna `id` e 5 linhas, sem ler nenhum arquivo.

In [ ]:
# crie um DataFrame com 5 linhas usando spark.range e mostre


### Os dados do projeto

In [ ]:
# liste o conteúdo da pasta ../../data


| Pasta | O que tem |
|---|---|
| `raw` | o histórico das Copas em três CSVs: as edições, as partidas e as escalações |
| `events` | o feed de eventos: um registro para cada lance de cada partida |
| `raw_2026` | os dados da Copa de 2026, no mesmo formato do histórico |
| `events_2026` | o feed de eventos da Copa de 2026 |

O `README.md` da pasta descreve as colunas de cada conjunto.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()